In [18]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

In [23]:
model_name = "Qwen/Qwen1.5-0.5B-Chat"


In [24]:
# ===== Load Model (4bit) =====
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


OutOfMemoryError: CUDA out of memory. Tried to allocate 298.00 MiB. GPU 0 has a total capacity of 1.64 GiB of which 139.31 MiB is free. Including non-PyTorch memory, this process has 1.50 GiB memory in use. Of the allocated memory 1.04 GiB is allocated by PyTorch, and 411.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:





tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# ===== Load Dataset =====
dataset = load_dataset("json", data_files="train.json")


# ===== Format Data =====
def format(example):
    example["text"] = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n{example['output']}"
    )
    return example

dataset = dataset.map(format)


# ===== Tokenization =====
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    # Labels = same as input_ids (causal LM)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, batched=True)


# ===== LoRA Config =====
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)


# ===== Data Collator =====
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


# ===== Training =====
training_args = TrainingArguments(
    output_dir="./finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=20,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    fp16=True,
    remove_unused_columns=False,   # مهم جدًا!
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=data_collator,
)

trainer.train()


# ===== Save =====
model.save_pretrained("finetuned_lora")
tokenizer.save_pretrained("finetuned_lora")
